In [176]:
import pandas as pd
import numpy as np

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")

print("Orders:", orders.shape)
print("Customers:", customers.shape)
print("Order Items:", order_items.shape)
print("Payments:", payments.shape)
print("Products:", products.shape)
print("Sellers:", sellers.shape)

Orders: (99441, 8)
Customers: (99441, 5)
Order Items: (112650, 7)
Payments: (103886, 5)
Products: (32951, 9)
Sellers: (3095, 4)


In [177]:
# Basic information about the orders dataset

orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [178]:
# Convert date columns to datetime

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [179]:
# Keep delivered orders only
model_data = orders[orders["order_status"] == "delivered"].copy()

# Remove orders without actual delivery date
model_data = model_data.dropna(
    subset=["order_delivered_customer_date"]
)

# Create the target
model_data["delivered_late"] = (
    model_data["order_delivered_customer_date"]
    > model_data["order_estimated_delivery_date"]
).astype(int)

print("Usable orders:", len(model_data))
print("\nTarget distribution:")
print(model_data["delivered_late"].value_counts())

print("\nTarget percentage:")
print(
    (model_data["delivered_late"].value_counts(normalize=True) * 100)
    .round(2)
)

Usable orders: 96470

Target distribution:
delivered_late
0    88644
1     7826
Name: count, dtype: int64

Target percentage:
delivered_late
0    91.89
1     8.11
Name: proportion, dtype: float64


In [180]:
# Create delivery planning feature

model_data["estimated_delivery_days"] = (
    model_data["order_estimated_delivery_date"]
    - model_data["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

print(
    model_data["estimated_delivery_days"].describe()
)

count    96470.000000
mean        23.736343
std          8.761052
min          2.008009
25%         18.329905
50%         23.230880
75%         28.407795
max        155.135463
Name: estimated_delivery_days, dtype: float64


In [181]:
# Create time-based features from the order purchase timestamp

model_data["purchase_year"] = model_data["order_purchase_timestamp"].dt.year
model_data["purchase_month"] = model_data["order_purchase_timestamp"].dt.month
model_data["purchase_day"] = model_data["order_purchase_timestamp"].dt.day
model_data["purchase_dayofweek"] = model_data["order_purchase_timestamp"].dt.dayofweek
model_data["purchase_hour"] = model_data["order_purchase_timestamp"].dt.hour

# Weekend indicator
model_data["is_weekend"] = (
    model_data["purchase_dayofweek"] >= 5
).astype(int)

model_data[
    [
        "order_purchase_timestamp",
        "purchase_year",
        "purchase_month",
        "purchase_dayofweek",
        "purchase_hour",
        "is_weekend",
        "delivered_late"
    ]
].head()

,order_purchase_timestamp,purchase_year,purchase_month,purchase_dayofweek,purchase_hour,is_weekend,delivered_late
0,2017-10-02 10:56:33,2017,10,0,10,0,0
1,2018-07-24 20:41:37,2018,7,1,20,0,0
2,2018-08-08 08:38:49,2018,8,2,8,0,0
3,2017-11-18 19:28:06,2017,11,5,19,1,0
4,2018-02-13 21:18:39,2018,2,1,21,0,0


In [182]:
# Aggregate order-item information at the order level

order_item_features = (
    order_items
    .groupby("order_id")
    .agg(
        total_items=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        average_item_price=("price", "mean")
    )
    .reset_index()
)

order_item_features.head()

,order_id,total_items,total_price,total_freight,average_item_price
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,58.90
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,239.90
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,199.00
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,199.90


In [183]:
# Inspect seller data

print(sellers.shape)
print(sellers.columns.tolist())
sellers.head()

(3095, 4)
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [184]:
# Create seller-related features at the order level

order_seller_features = (
    order_items
    .groupby("order_id")
    .agg(
        unique_sellers=("seller_id", "nunique")
    )
    .reset_index()
)

# Add seller state information
order_items_with_seller = order_items.merge(
    sellers[["seller_id", "seller_state"]],
    on="seller_id",
    how="left"
)

seller_state_features = (
    order_items_with_seller
    .groupby("order_id")
    .agg(
        unique_seller_states=("seller_state", "nunique")
    )
    .reset_index()
)

# Combine seller features
order_seller_features = order_seller_features.merge(
    seller_state_features,
    on="order_id",
    how="left"
)

order_seller_features.head()

,order_id,unique_sellers,unique_seller_states
0,00010242fe8c5a6d1ba2dd792cb16214,1,1
1,00018f77f2f0320c557190d7a144bdd3,1,1
2,000229ec398224ef6ca0657da4fc703e,1,1
3,00024acbcdf0a6daa1e931b038114c75,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1


In [185]:
# Historical seller performance features

# Keep only one row per order-seller combination
order_seller = (
    order_items[["order_id", "seller_id"]]
    .drop_duplicates()
)

seller_history = (
    model_data[
        [
            "order_id",
            "order_purchase_timestamp",
            "delivered_late"
        ]
    ]
    .merge(
        order_seller,
        on="order_id",
        how="left"
    )
)

seller_history = seller_history.sort_values(
    ["seller_id", "order_purchase_timestamp", "order_id"]
)

# Number of previous orders handled by the seller
seller_history["seller_previous_orders"] = (
    seller_history.groupby("seller_id").cumcount()
)

# Number of previous late orders
seller_history["seller_previous_late"] = (
    seller_history.groupby("seller_id")["delivered_late"].cumsum()
    - seller_history["delivered_late"]
)

# Historical late-delivery rate
seller_history["seller_previous_late_rate"] = np.where(
    seller_history["seller_previous_orders"] > 0,
    seller_history["seller_previous_late"]
    / seller_history["seller_previous_orders"],
    np.nan
)

print("Seller history shape:", seller_history.shape)
print("Unique orders:", seller_history["order_id"].nunique())
print("Unique sellers:", seller_history["seller_id"].nunique())

Seller history shape: (97811, 7)
Unique orders: 96470
Unique sellers: 2970


In [186]:
# Aggregate seller history to one row per order

seller_features = (
    seller_history
    .groupby("order_id")
    .agg(
        seller_previous_orders=("seller_previous_orders", "max"),
        seller_previous_late=("seller_previous_late", "max"),
        seller_previous_late_rate=("seller_previous_late_rate", "max")
    )
    .reset_index()
)

print("Seller features shape:", seller_features.shape)
print("Unique orders:", seller_features["order_id"].nunique())
print(seller_features.head())

Seller features shape: (96470, 4)
Unique orders: 96470
                           order_id  seller_previous_orders  \
0  00010242fe8c5a6d1ba2dd792cb16214                      83   
1  00018f77f2f0320c557190d7a144bdd3                       0   
2  000229ec398224ef6ca0657da4fc703e                       6   
3  00024acbcdf0a6daa1e931b038114c75                      10   
4  00042b26cf59d7ce69dfabb4e55b4fd9                       3   

   seller_previous_late  seller_previous_late_rate  
0                     4                   0.048193  
1                     0                        NaN  
2                     0                   0.000000  
3                     1                   0.100000  
4                     0                   0.000000  


In [187]:
# Create customer-seller geographic feature

customer_location = customers[
    ["customer_id", "customer_state"]
].copy()

seller_location = sellers[
    ["seller_id", "seller_state"]
].copy()

# Connect order items with customer and seller locations
geo_data = (
    orders[["order_id", "customer_id"]]
    .merge(customer_location, on="customer_id", how="left")
    .merge(
        order_items[["order_id", "seller_id"]],
        on="order_id",
        how="left"
    )
    .merge(seller_location, on="seller_id", how="left")
)

# Determine whether customer and seller are in the same state
geo_data["same_state"] = (
    geo_data["customer_state"] == geo_data["seller_state"]
).astype(int)

# Aggregate to order level
geo_features = (
    geo_data
    .groupby("order_id")
    .agg(
        same_state=("same_state", "max")
    )
    .reset_index()
)

geo_features.head()

,order_id,same_state
0,00010242fe8c5a6d1ba2dd792cb16214,0
1,00018f77f2f0320c557190d7a144bdd3,1
2,000229ec398224ef6ca0657da4fc703e,1
3,00024acbcdf0a6daa1e931b038114c75,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,0


In [188]:
# Inspect payment data

print(payments.shape)
print(payments.columns.tolist())
payments.head()

(103886, 5)
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [189]:
# Create payment-related features at the order level

payment_features = (
    payments
    .groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_methods=("payment_type", "nunique"),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

payment_features.head()

,order_id,total_payment_value,payment_methods,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3


In [190]:
# Inspect product data

print(products.shape)
print(products.columns.tolist())
products.head()

(32951, 9)
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [191]:
# Create product-related features at the order level

order_products = order_items.merge(
    products[
        [
            "product_id",
            "product_category_name",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm",
            "product_photos_qty"
        ]
    ],
    on="product_id",
    how="left"
)

# Calculate product volume
order_products["product_volume_cm3"] = (
    order_products["product_length_cm"]
    * order_products["product_height_cm"]
    * order_products["product_width_cm"]
)

# Aggregate product characteristics by order
product_features = (
    order_products
    .groupby("order_id")
    .agg(
        unique_product_categories=("product_category_name", "nunique"),
        average_product_weight_g=("product_weight_g", "mean"),
        total_product_weight_g=("product_weight_g", "sum"),
        average_product_volume_cm3=("product_volume_cm3", "mean"),
        total_product_volume_cm3=("product_volume_cm3", "sum"),
        average_product_photos=("product_photos_qty", "mean")
    )
    .reset_index()
)

product_features.head()

,order_id,unique_product_categories,average_product_weight_g,total_product_weight_g,average_product_volume_cm3,total_product_volume_cm3,average_product_photos
0,00010242fe8c5a6d1ba2dd792cb16214,1,650.0,650.0,3528.0,3528.0,4.0
1,00018f77f2f0320c557190d7a144bdd3,1,30000.0,30000.0,60000.0,60000.0,2.0
2,000229ec398224ef6ca0657da4fc703e,1,3050.0,3050.0,14157.0,14157.0,2.0
3,00024acbcdf0a6daa1e931b038114c75,1,200.0,200.0,2400.0,2400.0,1.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,3750.0,3750.0,42000.0,42000.0,1.0


In [192]:
# Combine all engineered features

feature_data = model_data[
    [
        "order_id",
        "customer_id",
        "purchase_year",
        "purchase_month",
        "purchase_day",
        "purchase_dayofweek",
        "purchase_hour",
        "is_weekend",
        "estimated_delivery_days",
        "delivered_late"
    ]
].copy()

feature_data = feature_data.merge(
    order_item_features,
    on="order_id",
    how="left"
)

feature_data = feature_data.merge(
    order_seller_features,
    on="order_id",
    how="left"
)

feature_data = feature_data.merge(
    geo_features,
    on="order_id",
    how="left"
)

feature_data = feature_data.merge(
    payment_features,
    on="order_id",
    how="left"
)

feature_data = feature_data.merge(
    product_features,
    on="order_id",
    how="left"
)

print("Final feature dataset shape:", feature_data.shape)

feature_data.head()

Final feature dataset shape: (96470, 26)


,order_id,customer_id,purchase_year,purchase_month,purchase_day,purchase_dayofweek,purchase_hour,is_weekend,estimated_delivery_days,delivered_late,...,same_state,total_payment_value,payment_methods,max_installments,unique_product_categories,average_product_weight_g,total_product_weight_g,average_product_volume_cm3,total_product_volume_cm3,average_product_photos
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017,10,2,0,10,0,15.544063,0,...,1,38.71,2.0,1.0,1,500.0,500.0,1976.0,1976.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018,7,24,1,20,0,19.137766,0,...,0,141.46,1.0,1.0,1,400.0,400.0,4693.0,4693.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018,8,8,2,8,0,26.639711,0,...,0,179.12,1.0,3.0,1,420.0,420.0,9576.0,9576.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017,11,18,5,19,1,26.188819,0,...,0,72.20,1.0,1.0,1,450.0,450.0,6000.0,6000.0,3.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018,2,13,1,21,0,12.112049,0,...,1,28.62,1.0,1.0,1,250.0,250.0,11475.0,11475.0,4.0


In [193]:
# Merge historical seller features

feature_data = feature_data.merge(
    seller_features,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Orders with no previous seller history
feature_data["seller_previous_orders"] = (
    feature_data["seller_previous_orders"].fillna(0)
)

feature_data["seller_previous_late"] = (
    feature_data["seller_previous_late"].fillna(0)
)

feature_data["seller_previous_late_rate"] = (
    feature_data["seller_previous_late_rate"].fillna(0)
)

print("Feature dataset after seller history:", feature_data.shape)
print(
    feature_data[
        [
            "seller_previous_orders",
            "seller_previous_late",
            "seller_previous_late_rate"
        ]
    ].describe()
)

Feature dataset after seller history: (96470, 29)
       seller_previous_orders  seller_previous_late  seller_previous_late_rate
count            96470.000000          96470.000000               96470.000000
mean               186.265222             14.305712                   0.067134
std                299.864928             27.129874                   0.083726
min                  0.000000              0.000000                   0.000000
25%                 16.000000              0.000000                   0.000000
50%                 60.000000              3.000000                   0.051136
75%                204.000000             13.000000                   0.092593
max               1818.000000            195.000000                   1.000000


In [194]:
# Validate seller history features

print("Missing seller history values:")
print(
    feature_data[
        [
            "seller_previous_orders",
            "seller_previous_late",
            "seller_previous_late_rate"
        ]
    ].isna().sum()
)

print("\nDuplicate order IDs:")
print(feature_data["order_id"].duplicated().sum())

print("\nSeller history sample:")
print(
    feature_data[
        [
            "order_id",
            "seller_previous_orders",
            "seller_previous_late",
            "seller_previous_late_rate"
        ]
    ].head(10)
)

Missing seller history values:
seller_previous_orders       0
seller_previous_late         0
seller_previous_late_rate    0
dtype: int64

Duplicate order IDs:
0

Seller history sample:
                           order_id  seller_previous_orders  \
0  e481f51cbdc54678b7cc49136f2d6af7                      41   
1  53cdb2fc8bc7dce0b6741e2150273451                      59   
2  47770eb9100c2d0c44946d9cf07ec65d                    1079   
3  949d5b44dbf5de918fe9c16f97b45f8a                      37   
4  ad21c59c0840e6cb83a9ceb5573f8159                      42   
5  a4591c265e18cb1dcee52889e2d8acc3                     142   
6  6514b8ad8028c9f2cc2374ded245783f                      15   
7  76c6e866289321a7c93b82b54852dc33                       1   
8  e69bfb5eb88e0ed6a785585b27e16dbf                     262   
9  e6ce16cb79ec1d90b1da9085a6118aeb                      40   

   seller_previous_late  seller_previous_late_rate  
0                     0                   0.000000  
1              

In [195]:
# Check all missing values

missing_values = feature_data.isna().sum()

print("Columns with missing values:")
print(missing_values[missing_values > 0])

Columns with missing values:
total_payment_value              1
payment_methods                  1
max_installments                 1
average_product_weight_g        16
average_product_volume_cm3      16
average_product_photos        1332
dtype: int64


In [196]:
# Handle remaining missing values

# Payment-related missing values
feature_data["total_payment_value"] = (
    feature_data["total_payment_value"].fillna(0)
)

feature_data["payment_methods"] = (
    feature_data["payment_methods"].fillna("unknown")
)

feature_data["max_installments"] = (
    feature_data["max_installments"].fillna(0)
)

# Product-related missing values
feature_data["average_product_weight_g"] = (
    feature_data["average_product_weight_g"]
    .fillna(feature_data["average_product_weight_g"].median())
)

feature_data["average_product_volume_cm3"] = (
    feature_data["average_product_volume_cm3"]
    .fillna(feature_data["average_product_volume_cm3"].median())
)

feature_data["average_product_photos"] = (
    feature_data["average_product_photos"]
    .fillna(feature_data["average_product_photos"].median())
)

print("Missing values after imputation:")
print(feature_data.isna().sum()[feature_data.isna().sum() > 0])

Missing values after imputation:
Series([], dtype: int64)


In [197]:
# Check for duplicate orders

duplicate_orders = feature_data["order_id"].duplicated().sum()

print("Duplicate order IDs:", duplicate_orders)
print("Unique order IDs:", feature_data["order_id"].nunique())
print("Total rows:", len(feature_data))

Duplicate order IDs: 0
Unique order IDs: 96470
Total rows: 96470


In [198]:
# Final feature dataset check

print("Final dataset shape:", feature_data.shape)

print("\nDuplicate order IDs:")
print(feature_data["order_id"].duplicated().sum())

print("\nTarget distribution:")
print(feature_data["delivered_late"].value_counts())

print("\nFinal columns:")
print(feature_data.columns.tolist())

Final dataset shape: (96470, 29)

Duplicate order IDs:
0

Target distribution:
delivered_late
0    88644
1     7826
Name: count, dtype: int64

Final columns:
['order_id', 'customer_id', 'purchase_year', 'purchase_month', 'purchase_day', 'purchase_dayofweek', 'purchase_hour', 'is_weekend', 'estimated_delivery_days', 'delivered_late', 'total_items', 'total_price', 'total_freight', 'average_item_price', 'unique_sellers', 'unique_seller_states', 'same_state', 'total_payment_value', 'payment_methods', 'max_installments', 'unique_product_categories', 'average_product_weight_g', 'total_product_weight_g', 'average_product_volume_cm3', 'total_product_volume_cm3', 'average_product_photos', 'seller_previous_orders', 'seller_previous_late', 'seller_previous_late_rate']


In [199]:
# Sort orders chronologically
feature_data = feature_data.sort_values("purchase_year").copy()

# Create a proper purchase date for splitting
feature_data["purchase_date"] = pd.to_datetime(
    feature_data[
        ["purchase_year", "purchase_month", "purchase_day"]
    ].rename(
        columns={
            "purchase_year": "year",
            "purchase_month": "month",
            "purchase_day": "day"
        }
    )
)

# Sort by the complete purchase date
feature_data = feature_data.sort_values("purchase_date").reset_index(drop=True)

# Use the first 80% for training and the latest 20% for testing
split_index = int(len(feature_data) * 0.80)

train_data = feature_data.iloc[:split_index].copy()
test_data = feature_data.iloc[split_index:].copy()

print("Training rows:", len(train_data))
print("Testing rows:", len(test_data))

print("\nTraining period:")
print(train_data["purchase_date"].min(), "to", train_data["purchase_date"].max())

print("\nTesting period:")
print(test_data["purchase_date"].min(), "to", test_data["purchase_date"].max())

Training rows: 77176
Testing rows: 19294

Training period:
2016-09-15 00:00:00 to 2018-05-26 00:00:00

Testing period:
2018-05-26 00:00:00 to 2018-08-29 00:00:00


In [200]:
# Save final modeling dataset

output_path = "../data/processed/modeling_dataset.csv"

feature_data.to_csv(output_path, index=False)

print("Final modeling dataset saved successfully!")
print("Path:", output_path)
print("Shape:", feature_data.shape)

Final modeling dataset saved successfully!
Path: ../data/processed/modeling_dataset.csv
Shape: (96470, 30)
